
## Misión Asíncrona: Cazando al Falso Influencer

### Fase 1 : Data Engineering

Para poder usar networkx, primero necesitan una Lista de Aristas purificada. Abran un nuevo Jupyter Notebook, importen pandas y realicen la siguiente tubería de datos (Data Pipeline):

- Limpieza Estructural: Eliminen las filas donde la columna Cuentas_Que_Sigue esté vacía (nula). Estandaricen la columna Usuario a MAYÚSCULAS y quítenle los espacios al inicio/final. Eliminen usuarios duplicados.
- La Explosión: Separen los nombres de la columna Cuentas_Que_Sigue usando .str.split(','). Luego, utilicen la función mágica de Pandas .explode() sobre esa columna para aplanar la tabla y convertirla en una verdadera lista de Origen -> Destino.
- Purificación Post-Explosión: Vuelvan a limpiar los textos de la columna destino (mayúsculas y sin espacios extra). Finalmente, eliminen los Self-Loops (Registros donde el Usuario aparezca siguiéndose a sí mismo).

In [8]:
## Import the data into df variable
import pandas as pd

df = pd.read_csv('export_api_influencers.csv')

print(df.head(10))

  Usuario Ubicacion Fecha_Creacion Cuentas_Que_Sigue
0   Bot_5     India     2023-05-06         AMATEUR_1
1   fan_6    Mexico     2023-07-14   GARNACHO, MESSI
2  Bot_11     India     2023-02-14         AMATEUR_1
3   Bot_1     India     2023-11-24         AMATEUR_1
4  Bot_36     India     2023-06-24         AMATEUR_1
5   Fan_4    Mexico     2023-05-29          GARNACHO
6  Bot_29     India     2023-08-18         AMATEUR_1
7  Bot_13     India     2023-08-05         AMATEUR_1
8  Bot_18     India     2023-04-30         AMATEUR_1
9   Fan_7    Mexico     2023-11-22          GARNACHO


In [9]:
### Limpieza Estructural

## 1. Delete rows when Cuentas_que_sigue is empty
df = df.dropna(subset=['Cuentas_Que_Sigue'])

## 2. Remove duplicates
df = df.drop_duplicates(subset=['Cuentas_Que_Sigue'])

## 3. Standarized the User colum to all uppercase and delete spaces at the start and end
df['Usuario'] = df['Usuario'].astype(str).str.strip().str.upper()


print("Valores nulos : ", df['Cuentas_Que_Sigue'].isna().sum())
print("Duplicados : ", df['Cuentas_Que_Sigue'].duplicated().sum())  # Debe devolver 0
display(df.head(5))

Valores nulos :  0
Duplicados :  0


,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,BOT_5,India,2023-05-06,AMATEUR_1
1,FAN_6,Mexico,2023-07-14,"GARNACHO, MESSI"
5,FAN_4,Mexico,2023-05-29,GARNACHO
18,PAREDES,Argentina/Brasil,2023-11-18,"MESSI, DI_MARIA, SUAREZ, NEYMAR, OTAMENDI"
30,DI_MARIA,Argentina/Brasil,2023-02-10,"DE_PAUL, SUAREZ, NEYMAR, OTAMENDI"


In [10]:
## Explosion
df['Cuentas_Que_Sigue'] = df['Cuentas_Que_Sigue'].astype(str).str.split(',')
df = df.explode('Cuentas_Que_Sigue')

display(df.head(10))

,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,BOT_5,India,2023-05-06,AMATEUR_1
1,FAN_6,Mexico,2023-07-14,GARNACHO
1,FAN_6,Mexico,2023-07-14,MESSI
5,FAN_4,Mexico,2023-05-29,GARNACHO
18,PAREDES,Argentina/Brasil,2023-11-18,MESSI
18,PAREDES,Argentina/Brasil,2023-11-18,DI_MARIA
18,PAREDES,Argentina/Brasil,2023-11-18,SUAREZ
18,PAREDES,Argentina/Brasil,2023-11-18,NEYMAR
18,PAREDES,Argentina/Brasil,2023-11-18,OTAMENDI
30,DI_MARIA,Argentina/Brasil,2023-02-10,DE_PAUL


In [11]:
## Post - Explotion
df['Cuentas_Que_Sigue'] = df['Cuentas_Que_Sigue'].astype(str).str.strip().str.upper()

df = df[df['Usuario'] != df['Cuentas_Que_Sigue']]

### Fase 2 : El engaño de la popularidad

- Importen networkx y conviertan su tabla en un Grafo Dirigido (nx.DiGraph).
- Calculen la Centralidad de Entrada (nx.in_degree_centrality).
- Impriman a los 3 usuarios con el puntaje más alto.


In [15]:
import networkx as nx
import matplotlib.pyplot as plt

G_graph = nx.from_pandas_edgelist(
    df,
    source='Usuario',
    target='Cuentas_Que_Sigue',
    create_using=nx.DiGraph()
)
centrality_in_degree = nx.in_degree_centrality(G_graph)

df_resultados = pd.DataFrame({
    'Seguidores_Reales': dict(G_graph.in_degree()),
    'Centralidad_In_Degree': centrality_in_degree
}).sort_values(by='Centralidad_In_Degree', ascending=False)

display(df_resultados.head(10))


,Seguidores_Reales,Centralidad_In_Degree
SUAREZ,6,0.545455
DI_MARIA,6,0.545455
GARNACHO,5,0.454545
NEYMAR,5,0.454545
OTAMENDI,5,0.454545
DE_PAUL,5,0.454545
MESSI,5,0.454545
PAREDES,3,0.272727
AMATEUR_1,1,0.090909
BOT_5,0,0.000000


Pregunta: A simple vista, ¿quién parece ser el rey de la red?

Se podrioa decir que Suarez y Di maria son los que tienen mas seguidores, despues aparece GARNACHO

### Fase 3 : La Verdad Topológica (Las Métricas)

- Calculen el PageRank (nx.pagerank). ¿Quién tiene ahora la verdadera autoridad? Se posicionan Suares, Otamendi y Di Maria, mientras que Garnacho decendio muhcos lugares por lo que se podria decir que tenia muchos bots
- Conviertan el grafo a No Dirigido y calculen el Coeficiente de Clustering (nx.clustering). Impriman el valor exacto de AMATEUR_1 y el de GARNACHO.

In [16]:
pagerank = nx.pagerank(G_graph)

df_pagerank = pd.DataFrame.from_dict(
    pagerank, orient='index', columns=['PageRank']
).sort_values(by='PageRank', ascending=False)

print("--- Top 10 PageRank ---")
display(df_pagerank.head(10))

--- Top 10 PageRank ---


,PageRank
SUAREZ,0.143588
OTAMENDI,0.133995
DI_MARIA,0.131646
DE_PAUL,0.129009
NEYMAR,0.118375
GARNACHO,0.099395
MESSI,0.094312
PAREDES,0.079913
AMATEUR_1,0.026612
BOT_5,0.014385


In [14]:
G_undirected = G_graph.to_undirected()

clustering_scores = nx.clustering(G_undirected)

print("\n--- Coeficiente de Clustering ---")
print("AMATEUR_1:", clustering_scores.get("AMATEUR_1"))
print("GARNACHO:", clustering_scores.get("GARNACHO"))


--- Coeficiente de Clustering ---
AMATEUR_1: 0
GARNACHO: 0.47619047619047616


¿Quién compró bots y por qué?
El influencer AMATEUR_1 tuvo un in_degree de 0.090909, lo que se considera alto, pero al compararlo con le pagerank de 0.026612, se nota un bajon drastico, dictando que sus seguridores no estan activos o no son relevantes dentro de la plataforma, ademas que por el valor del clustering (0.0) nos dice que sus seguidores no interactuan entre si,  por lo que se puede inferir como bots

¿Quién es el influencer real y por qué?
GARNACHO, que aunque tambien experimento un bajon en la comparacion de las metricas, se justifica ya que cuenta con relaciones con otros futbolistas como MESSI, DI MARIA Y OTAMENDI, y estos al ser usuarios activos y interactuan entre si (clustering = 0.47) respaldan la veracidad de sus seguidores siendo la opcion mas confiable.